In [2]:
# pipenv install pandas scipy optuna shap plotly scikit-learn matplotlib numpy ipywidgets ipykernel nbformat

# EDA
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import plotly.express as px

# ML
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss, roc_curve, roc_auc_score, f1_score, precision_score

# Hyperparams optimization
import optuna

# Data Preparation

In [14]:
# Load dataset
df_churn = pd.read_csv('./datasets/churn_employees_dataset.csv', 
                       parse_dates=['hire_date', 'termination_date', 'last_feedback_date', 'last_raise_date', 'last_job_change_date'],
                       date_format='%Y-%m-%d')
df_churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   id                          2000 non-null   str           
 1   age                         2000 non-null   int64         
 2   gender                      2000 non-null   str           
 3   marital_status              2000 non-null   str           
 4   education                   2000 non-null   str           
 5   employment_status           2000 non-null   str           
 6   hire_date                   2000 non-null   datetime64[us]
 7   termination_date            286 non-null    datetime64[us]
 8   termination_type            286 non-null    str           
 9   job_title                   2000 non-null   str           
 10  current_salary              2000 non-null   int64         
 11  last_feedback_date          2000 non-null   datetime64[us]
 12  las

In [15]:
df_churn.head(10)

,id,age,gender,marital_status,education,employment_status,hire_date,termination_date,termination_type,job_title,current_salary,last_feedback_date,last_raise_date,last_job_change_date,evaluation_score,psychological_counseling,number_of_projects,number_of_clients,manager_satisfaction_level,churn
0,EMP1564,37,F,Divorced,Master,Remote,2020-01-02,NaT,NaN,Senior Developer,10207,2024-05-09,2023-07-31,2022-11-03,9.6,True,5,1,7.0,0
1,EMP0959,45,Other,Divorced,Master,Remote,2020-01-03,NaT,NaN,UX Designer,23921,2024-04-05,2023-09-10,2022-02-11,8.4,True,8,4,9.1,0
2,EMP0494,54,M,Widower/Widow,Technical Degree,Hybrid,2020-01-04,NaT,NaN,Tech Lead,15298,2023-06-16,2021-12-26,2021-05-10,8.5,False,9,7,1.3,0
3,EMP1231,44,M,Married,Bachelor,Remote,2020-01-05,NaT,NaN,QA Engineer,15306,2021-12-04,2022-01-01,2024-08-30,7.9,False,8,4,8.4,0
4,EMP1912,53,Other,Widower/Widow,PhD,In-person,2020-01-07,NaT,NaN,Senior Developer,17723,2022-03-04,2023-11-18,2024-08-08,7.5,True,5,2,8.2,0
5,EMP1053,49,Other,Married,Technical Degree,Hybrid,2020-01-08,NaT,NaN,Junior Developer,6507,2022-06-01,2020-07-05,2021-09-02,7.9,True,8,4,7.1,0
6,EMP1152,29,M,Married,PhD,In-person,2020-01-10,2022-08-22,Involuntary,Product Manager,12575,2020-10-16,2020-08-11,2022-02-27,9.1,True,6,3,1.9,1
7,EMP1317,38,F,Widower/Widow,PhD,Remote,2020-01-11,NaT,NaN,DevOps Engineer,16643,2023-03-28,2023-05-16,2022-03-28,8.9,False,6,4,8.3,0
8,EMP1398,33,M,Married,Technical Degree,Remote,2020-01-12,NaT,NaN,DevOps Engineer,16771,2024-01-03,2024-03-13,2024-06-15,7.6,False,8,1,1.2,0
9,EMP0167,23,M,Widower/Widow,PhD,Hybrid,2020-01-13,NaT,NaN,QA Engineer,13258,2022-01-02,2021-09-15,2024-04-04,8.1,True,3,6,7.0,0


## Feature Engineering

In [18]:
# Creating features based on dates

# Calculate the time at company
df_churn['time_at_company'] = df_churn.apply(lambda x: 
                                                (pd.Timestamp.now() - x['hire_date']).days if x['churn'] == 0
                                                else (x['termination_date'] - x['hire_date']).days, axis=1
                                             )

# Calculate time since last feedback
df_churn['time_since_last_feedback'] = df_churn.apply(lambda x: 
                                                         (pd.Timestamp.now() - x['last_feedback_date']).days, axis=1
                                                      )

# Calculate time since last raise
df_churn['time_since_last_raise'] = df_churn.apply(lambda x: 
                                                         (pd.Timestamp.now() - x['last_raise_date']).days, axis=1
                                                      )

# Calculate time since last job change
df_churn['time_since_last_job_change'] = df_churn.apply(lambda x: 
                                                         (pd.Timestamp.now() - x['last_job_change_date']).days, axis=1
                                                      )

df_churn.head(2)

,id,age,gender,marital_status,education,employment_status,hire_date,termination_date,termination_type,job_title,...,evaluation_score,psychological_counseling,number_of_projects,number_of_clients,manager_satisfaction_level,churn,time_at_company,time_since_last_feedback,time_since_last_raise,time_since_last_job_change
0,EMP1564,37,F,Divorced,Master,Remote,2020-01-02,NaT,NaN,Senior Developer,...,9.6,True,5,1,7.0,0,2374,785,1068,1338
1,EMP0959,45,Other,Divorced,Master,Remote,2020-01-03,NaT,NaN,UX Designer,...,8.4,True,8,4,9.1,0,2373,819,1027,1603


In [20]:
df_churn.drop(columns=['id'], inplace=True)

## EDA

In [22]:
print("Null countdown per column")
df_churn.isnull().sum()

Null countdown per column


age                              0
gender                           0
marital_status                   0
education                        0
employment_status                0
hire_date                        0
termination_date              1714
termination_type              1714
job_title                        0
current_salary                   0
last_feedback_date               0
last_raise_date                  0
last_job_change_date             0
evaluation_score                 0
psychological_counseling         0
number_of_projects               0
number_of_clients                0
manager_satisfaction_level       0
churn                            0
time_at_company                  0
time_since_last_feedback         0
time_since_last_raise            0
time_since_last_job_change       0
dtype: int64

In [23]:
# Distribution target feature (Churn)

fig = px.bar(df_churn['churn'].value_counts() / len(df_churn) * 100,
             title='Churn Factor',
             labels={'index': 'Churn', 'value': 'Percentual'})

fig.update_layout(showlegend=False)
fig.show()

In [25]:
# Possible values for each categorical column
for col in df_churn.select_dtypes(include=['str']).columns:
    print(f'\nUnique Values in {col}:')
    print(f'{df_churn[col].unique()}')


Unique Values in gender:
<StringArray>
['F', 'Other', 'M']
Length: 3, dtype: str

Unique Values in marital_status:
<StringArray>
['Divorced', 'Widower/Widow', 'Married', 'Single']
Length: 4, dtype: str

Unique Values in education:
<StringArray>
['Master', 'Technical Degree', 'Bachelor', 'PhD']
Length: 4, dtype: str

Unique Values in employment_status:
<StringArray>
['Remote', 'Hybrid', 'In-person']
Length: 3, dtype: str

Unique Values in termination_type:
<StringArray>
[nan, 'Involuntary', 'Volunteer']
Length: 3, dtype: str

Unique Values in job_title:
<StringArray>
['Senior Developer',      'UX Designer',        'Tech Lead',
      'QA Engineer', 'Junior Developer',  'Product Manager',
  'DevOps Engineer',   'Data Scientist']
Length: 8, dtype: str


In [26]:
# Descritive stats for numeric features
df_churn.select_dtypes(include=['int64', 'float64']).describe()

,age,current_salary,evaluation_score,number_of_projects,number_of_clients,manager_satisfaction_level,churn,time_at_company,time_since_last_feedback,time_since_last_raise,time_since_last_job_change
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.00000,2000.000000,2000.000000
mean,37.938500,14801.834500,7.997550,4.984500,3.998500,5.492450,0.14300,1389.031000,1104.89900,1118.005000,1105.136500
std,9.456132,5763.451836,1.157324,2.593188,1.987076,2.609817,0.35016,589.426951,395.59628,397.623967,395.911115
min,22.000000,5016.000000,6.000000,1.000000,1.000000,1.000000,0.00000,34.000000,618.00000,619.000000,618.000000
25%,30.000000,9844.250000,7.000000,3.000000,2.000000,3.300000,0.00000,965.750000,774.75000,780.750000,778.750000
50%,38.000000,14822.500000,8.000000,5.000000,4.000000,5.500000,0.00000,1415.000000,1003.50000,1018.000000,995.000000
75%,46.000000,19702.500000,9.000000,7.000000,6.000000,7.700000,0.00000,1883.500000,1344.25000,1377.250000,1338.000000
max,54.000000,24988.000000,10.000000,9.000000,7.000000,10.000000,1.00000,2374.000000,2337.00000,2290.000000,2330.000000
